# OpenPlaque V21 — fresh 480/80 training control

Re-trains the **V9 architecture** from a new random initialization on the **v0.3 frozen positive-reference** split. This is a methodological control prior to testing nnU-Net/MONAI alternatives, **not** a claim of improved vessel specificity. Historical V9 trained on all 560 cases and is **not independent** of the new DEV80.

Run All with T4/L4 GPU. No manual coordinates, no source label changes, and no sealed test reads. Historical V7 partial-label screens remain enforced.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Reuse/cache controls. Keep the frozen split and existing V7 screen unchanged.
RUN_SMOKE = True
RUN_FULL_CACHE = True
RUN_FRESH_CONTROL_TRAINING = True
EPOCHS = 6
REUSE_VALID_CACHES = True


In [ ]:
import sys, subprocess, importlib.util, torch
missing=[p for p in ('nibabel','scipy','pandas') if importlib.util.find_spec(p) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())


In [ ]:
from pathlib import Path
import hashlib, json
root=Path('/content/drive/MyDrive/OpenPlaque')
old_nb=root/'OpenPlaque_ImageCASX_V9_NoHuman_UncertaintyAware_Research_Colab_20260928.ipynb'
assert old_nb.is_file(), 'The canonical V9 notebook is missing from the OpenPlaque Drive folder'
notebook=json.loads(old_nb.read_text(encoding='utf-8'))
src=[(''.join(c['source'])) for c in notebook['cells'] if c.get('cell_type')=='code' and ''.join(c['source']).startswith('%%writefile /content/openplaque_imagecasx_v9_autonomous_research.py')]
assert len(src)==1, 'Unexpected V9 notebook script layout'
source=src[0].split('\n',1)[1]
expected='f535e78a642f823d2e3b911609b2f3fd0438286a46c3f8af0129d4cc2fa6fdd1'
assert hashlib.sha256(source.encode()).hexdigest()==expected, 'V9 script fingerprint changed; stop'
source=source.replace("VERSION='op-imagecasx-v9-autonomous-research-20260928-1'","VERSION='op-imagecasx-v21-fresh-480-80-control-20261012-1'")
source=source.replace("OUT=BASE/'data_autonomous_research_v9'","FROZEN=DRIVE/'OpenPlaque_Evaluation_Dataset_v0_2'\nAUDIT=FROZEN/'sparse_topology_results_v0_3'\nOUT=FROZEN/'training_v21_fresh_control'")
source=source.replace("if len(train)<549 or len(val)<79:","if len(train)<475 or len(val)<79:")
source=source.replace("cached cases train {len(train)}/560 val {len(val)}/80","cached cases train {len(train)}/480 val {len(val)}/80")
source=source.replace("'cases_train':560,'cases_val':80","'cases_train':480,'cases_val':80")
source=source.replace("Strict source reconciliation: 640 cases; 8960 case×class rows.","Frozen V21: 480 new-model train and 80 internal development cases.")
override=r"""
# V21 override: reuse exact frozen V9 architecture/supervision but train on NEW frozen 480/80.
# Historical V9 saw all 560, so these results are not an independent comparison to V9.
_ORIGINAL_V9_LOAD_PLAN=load_plan
def load_plan():
    old_cases, old_member, old_plan_id=_ORIGINAL_V9_LOAD_PLAN()
    def read_ids(path):
        assert path.is_file(), 'Missing frozen split file: '+str(path)
        rows=[x.strip() for x in path.read_text().splitlines() if x.strip()]
        if len(rows)!=len(set(rows)) or not all(x.isdecimal() for x in rows):
            raise RuntimeError('Corrupted split: '+str(path))
        return set(rows)
    tr=read_ids(FROZEN/'train_480_frozen.txt')
    dv=read_ids(FROZEN/'internal_dev_80_frozen.txt')
    qa_path=AUDIT/'frozen_positive_reference_manifest_v0_3.json'
    assert qa_path.is_file(), 'No completed v0.3 audit'
    qa=json.loads(qa_path.read_text())
    if qa.get('status')!='FROZEN_POSITIVE_REFERENCE_QA_ONLY' or qa.get('audited_n')!=640:
        raise RuntimeError('v0.3 quality audit not complete')
    if qa.get('sealed_test_accessed') is not False or qa.get('source_masks_modified') is not False or qa.get('verified_negative_voxels')!=0:
        raise RuntimeError('Safety provenance failed')
    v19=json.loads((FROZEN/'source_v19_manifest.json').read_text())
    if len(tr)!=480 or len(dv)!=80 or tr&dv or tr|dv!=old_member['train']:
        raise RuntimeError('Split integrity failed: not exact original TRAIN560 re-split 480/80')
    if (tr|dv)&old_member['val']:
        raise RuntimeError('Historical validation leakage')
    if not set(map(str,v19['train_case_ids']))<=tr:
        raise RuntimeError('V19 training cases moved to new DEV')
    digest=hashlib.sha256(''.join(x+'\n' for x in sorted(dv,key=int)).encode()).hexdigest()
    if digest!=qa['frozen_dev_sha256']:
        raise RuntimeError('Frozen development SHA256 mismatch')
    cases={}
    for (old_split,cid), items in old_cases.items():
        if old_split!='train':continue
        new_split='train' if cid in tr else ('val' if cid in dv else None)
        if new_split is None:raise RuntimeError('Case missing V21 split assignment')
        cases[(new_split,cid)]=items
    if len(cases)!=560:raise RuntimeError('Expected 560 source train cases')
    plan=hashlib.sha256((VERSION+old_plan_id+digest+
        sha(FROZEN/'train_480_frozen.txt')+
        sha(FROZEN/'internal_dev_80_frozen.txt')+
        sha(qa_path)).encode()).hexdigest()
    print('V21 frozen 480/80 plan OK; sealed test untouched; V9 historical comparator NOT independent.')
    return cases, {'train':tr,'val':dv}, plan
"""
needle="if __name__=='__main__':main()"
assert source.count(needle)==1
source=source.replace(needle,override+'\n'+needle)
compile(source,'/content/openplaque_v21_control.py','exec')
script_path=Path('/content/openplaque_v21_control.py')
script_path.write_text(source,encoding='utf-8')
print('V9 source SHA verified:',expected)
print('V21 run script:',script_path,'SHA256:',hashlib.sha256(source.encode()).hexdigest())
print('Training data frozen: 480 train / 80 development; old validation diagnostic only')



In [ ]:
def stage(which,*args):
    command=[sys.executable,'/content/openplaque_v21_control.py',which,*map(str,args)]
    print('RUN',command,flush=True)
    subprocess.run(command,check=True)
stage('selftest')
stage('plan')


In [ ]:
if RUN_SMOKE:
    stage('cache','--split','train','--max-cases',3)
    stage('cache','--split','val','--max-cases',2)
if RUN_FULL_CACHE:
    stage('cache','--split','train')
    stage('cache','--split','val')


In [ ]:
if RUN_FRESH_CONTROL_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError('Select GPU in Colab Runtime settings; stop before model training')
    stage('train','--epochs',EPOCHS)
stage('report')


In [ ]:
from pathlib import Path
folder=Path('/content/drive/MyDrive/OpenPlaque/OpenPlaque_Evaluation_Dataset_v0_2/training_v21_fresh_control')
print('Saved outputs:',folder)
for name in ('train_status_v9.json','research_epoch_metrics_v9.json','v9_summary.json'):
    file=folder/name
    if file.exists(): print(name,file.read_text()[:7000])
print('Research-only partial-positive patch metrics; no verified negative truth and no sealed test reads.')
